# Stochastic Calculus from Zero · Episode 27: Jumps: Poisson processes and Merton

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Every path and every world in this notebook is simulated or made up, with fixed random seeds so the numbers repeat.
>
> **The share with jumps** (Series 3's share): priced at 100, a 3.60% risk-free rate, a call with strike 105 expiring in a year, a diffusion volatility of 15%, and on average 0.5 jumps a year whose log sizes are normal with mean −10% and standard deviation 15%. Made up for teaching.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. The Poisson process · 4. Merton's jump-diffusion · 5. One price, three ways · 6. The smile · 7. Hedging breaks · 8. Incomplete markets · 9. This episode's question · 10. Exercises · 11. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 27,
    "quantlib_version": ql.__version__,
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, math, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)


# ---- Part 2 additions -------------------------------------------------------------------------------------------
def riemann_sums(W, dW):
    """Sum of W dW sampled at the left end, the middle and the right end of each step (one path)."""
    return {"left": float((W[:-1] * dW).sum()), "mid": float((0.5 * (W[:-1] + W[1:]) * dW).sum()),
            "right": float((W[1:] * dW).sum())}


def ito_check(f, f1, f2, W, years=1.0):
    """For one path W on an even grid: the exact change f(W_T) - f(W_0), and what the school chain rule
    (sum of f'(W) dW) and the Ito formula (that plus 1/2 sum f''(W) dt) say it is."""
    dW, dt = np.diff(W), years / (len(W) - 1)
    chain = float((f1(W[:-1]) * dW).sum())
    return {"exact": float(f(W[-1]) - f(W[0])), "chain": chain, "ito": chain + 0.5 * float((f2(W[:-1]) * dt).sum())}


def correlated_paths(n_paths, steps, rho, rng, years=1.0):
    """Two Brownian motions with correlation rho, built from independent ones (Cholesky): returns (t, W1, W2)."""
    t, A, dA = brownian_paths(n_paths, steps, years, rng)
    _, _, dB = brownian_paths(n_paths, steps, years, rng)
    dW2 = rho * dA + np.sqrt(1 - rho ** 2) * dB
    return t, A, np.concatenate([np.zeros((n_paths, 1)), dW2.cumsum(axis=1)], axis=1)


def realised_vol(log_returns, years):
    """Annualised volatility from log returns: the square root of their sum of squares per year."""
    return np.sqrt((np.asarray(log_returns) ** 2).sum(axis=-1) / years)


# ---- Part 3 additions -------------------------------------------------------------------------------------------
#   Short rate (illustrative): r0 3.85%, long-run level 3.50%, mean reversion 0.30, volatility 1% (Vasicek).
def norm_cdf(x):
    """Standard normal distribution function, for numbers or numpy arrays."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(x) / math.sqrt(2)))


def bs_call(s, k, t, r, vol):
    """Black-Scholes call price and delta (Series 3, Episode 13)."""
    d1 = (np.log(s / k) + (r + 0.5 * vol ** 2) * t) / (vol * np.sqrt(t))
    return s * norm_cdf(d1) - k * np.exp(-r * t) * norm_cdf(d1 - vol * np.sqrt(t)), norm_cdf(d1)


def ou_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of an Ornstein-Uhlenbeck (Vasicek) rate t years ahead."""
    return theta + (r0 - theta) * math.exp(-a * t), sigma * math.sqrt((1 - math.exp(-2 * a * t)) / (2 * a))


def ou_step(r, dt, a, theta, sigma, z):
    """One exact OU step of length dt from r, driven by standard normals z: no discretisation error."""
    return theta + (r - theta) * math.exp(-a * dt) + sigma * math.sqrt((1 - math.exp(-2 * a * dt)) / (2 * a)) * z


def vasicek_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price P = A exp(-B r) in the Vasicek model, tau years to maturity."""
    B = (1 - math.exp(-a * tau)) / a
    lnA = (theta - sigma ** 2 / (2 * a * a)) * (B - tau) - sigma ** 2 * B * B / (4 * a)
    return np.exp(lnA - B * np.asarray(r))


def cir_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of a CIR rate t years ahead."""
    e = math.exp(-a * t)
    var = r0 * sigma ** 2 / a * (e - e * e) + theta * sigma ** 2 / (2 * a) * (1 - e) ** 2
    return theta + (r0 - theta) * e, math.sqrt(var)


def cir_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price in the CIR model (Cox, Ingersoll and Ross, 1985)."""
    g = math.sqrt(a * a + 2 * sigma * sigma)
    den = (g + a) * (math.exp(g * tau) - 1) + 2 * g
    B = 2 * (math.exp(g * tau) - 1) / den
    A_ = (2 * g * math.exp((a + g) * tau / 2) / den) ** (2 * a * theta / sigma ** 2)
    return A_ * np.exp(-B * np.asarray(r))


# ---- Part 4 additions -------------------------------------------------------------------------------------------
#   Hull-White fitted to today's curve P0(t), in the G1++ form r(t) = x(t) + phi(t), dx = -a x dt + sigma dW, x(0) = 0.
def hw_B(t, T, a):
    return (1 - np.exp(-a * (T - t))) / a


def hw_V(t, T, a, sigma):
    """Variance of the integral of x from t to T, given x(t)."""
    tau = T - t
    return sigma**2 / a**2 * (tau + 2 / a * np.exp(-a * tau) - np.exp(-2 * a * tau) / (2 * a) - 1.5 / a)


def hw_bond(t, T, x, P0, a, sigma):
    """Zero-coupon bond P(t, T) given the state x(t); P0 is today's discount curve."""
    return P0(T) / P0(t) * np.exp(0.5 * (hw_V(t, T, a, sigma) - hw_V(0, T, a, sigma) + hw_V(0, t, a, sigma))
                                  - hw_B(t, T, a) * np.asarray(x))


def hw_phi(t, f0, a, sigma):
    """The deterministic shift: r(t) = x(t) + phi(t) reprices today's curve (f0 = today's instantaneous forwards)."""
    return f0(t) + sigma**2 / (2 * a**2) * (1 - math.exp(-a * t))**2


def hw_exact_step(x, I, dt, a, sigma, z):
    """One exact step of (x, integral of x) over dt; z has shape (n, 2) of independent standard normals."""
    e = math.exp(-a * dt)
    bd = (1 - e) / a
    vx = sigma**2 / (2 * a) * (1 - e * e)
    vi = sigma**2 / a**2 * (dt - 2 * bd + (1 - e * e) / (2 * a))
    cxi = sigma**2 / (2 * a**2) * (1 - e)**2
    eps = z @ np.linalg.cholesky([[vx, cxi], [cxi, vi]]).T
    return x * e + eps[:, 0], I + x * bd + eps[:, 1]


#   Jumps: Merton's model, log jump sizes normal with mean mu_j and standard deviation d_j, lam jumps a year.
def merton_call(s, k, t, r, vol, lam, mu_j, d_j, terms=60):
    """Merton call price: Black-Scholes prices weighted by the Poisson chance of n jumps."""
    kbar = math.exp(mu_j + 0.5 * d_j**2) - 1
    lp = lam * (1 + kbar) * t
    total, w = 0.0, math.exp(-lp)
    for n in range(terms):
        if n:
            w *= lp / n
        total = total + w * bs_call(s, k, t, r - lam * kbar + n * math.log(1 + kbar) / t,
                                    math.sqrt(vol**2 + n * d_j**2 / t))[0]
    return total


def bs_implied_vol(price, s, k, t, r, lo=0.01, hi=1.5):
    """Black-Scholes implied volatility by bisection."""
    for _ in range(100):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if float(bs_call(s, k, t, r, mid)[0]) < price else (lo, mid)
    return 0.5 * (lo + hi)

## 2. Last episode's answer

With $\sigma(t, T) = \sigma$, the HJM drift is $\alpha = \sigma\int_t^T\sigma\,du = \sigma^2(T - t)$: the Ho-Lee model.

In [ ]:
out["prev"] = {"answer": "sigma^2 (T - t)"}

## 3. The Poisson process

A **Poisson process** $N_t$ counts events that arrive at a constant rate $\lambda$: it starts at 0, jumps up by 1 at random times, and its increments over disjoint intervals are independent with $N_{t+s} - N_t \sim \text{Poisson}(\lambda s)$. So

$$E[N_t] = \lambda t, \qquad \mathrm{Var}(N_t) = \lambda t, \qquad P(N_t = 0) = e^{-\lambda t}.$$

$N_t$ is not a martingale (it only goes up), but the **compensated** process $N_t - \lambda t$ is: subtract the expected count and what is left is a fair game. Its quadratic variation is $N_t$ itself (each jump of size 1 contributes $1^2$), not $t$: a martingale, but not a Brownian motion.

In [ ]:
S0, R, K, T = 100.0, 0.036, 105.0, 1.0
VOL, LAM, MUJ, DJ = 0.15, 0.5, -0.10, 0.15
KBAR = math.exp(MUJ + 0.5 * DJ**2) - 1          # average jump, J - 1
rng = np.random.default_rng(seed)
N5 = rng.poisson(LAM * 5, 200_000)
poisson = {"lam": LAM, "t": 5, "mean": float(N5.mean()), "var": float(N5.var()), "theory": LAM * 5,
           "comp_mean": float((N5 - LAM * 5).mean()), "comp_se": float(N5.std() / math.sqrt(N5.size)),
           "p_none_1y": math.exp(-LAM)}
assert abs(poisson["comp_mean"]) < 3 * poisson["comp_se"] and abs(poisson["var"] / poisson["theory"] - 1) < 0.02
# a few paths of N_t on a fine grid, for the picture
grid = np.linspace(0, 5, 501)
counts = np.cumsum(rng.poisson(LAM * (grid[1] - grid[0]), (12, len(grid) - 1)), axis=1)
poisson["t_grid"] = grid.round(3).tolist()
poisson["paths"] = np.concatenate([np.zeros((12, 1)), counts], axis=1).tolist()
poisson["mean_line"] = (LAM * grid).round(4).tolist()
poisson["ticks"] = [{"x": x, "label": f"{x}Y" if x else "0"} for x in range(6)]
out["poisson"] = poisson
{k: v for k, v in poisson.items() if not isinstance(v, list)}

## 4. Merton's jump-diffusion

Merton (1976) adds jumps to geometric Brownian motion. Under the risk-neutral measure,

$$\frac{dS}{S_{t-}} = (r - \lambda\bar k)\,dt + \sigma\,dW + (J - 1)\,dN, \qquad \ln J \sim N(\mu_J, \delta^2), \quad \bar k = E[J - 1] = e^{\mu_J + \frac12\delta^2} - 1 .$$

When a jump arrives the price is multiplied by $J$. The term $-\lambda\bar k\,dt$ is the **compensator**: jumps add $\lambda\bar k\,dt$ to the expected return, so the drift subtracts it and $e^{-rt}S_t$ stays a martingale. The solution is

$$S_T = S_0\exp\Big(\big(r - \lambda\bar k - \tfrac12\sigma^2\big)T + \sigma W_T + \sum_{i=1}^{N_T}\ln J_i\Big).$$

In [ ]:
n = 400_000
z = rng.standard_normal(n)
nj = rng.poisson(LAM * T, n)
jump = MUJ * nj + DJ * np.sqrt(nj) * rng.standard_normal(n)        # sum of nj normal log jumps
ST = S0 * np.exp((R - LAM * KBAR - 0.5 * VOL**2) * T + VOL * math.sqrt(T) * z + jump)
# 30 daily paths for the picture
steps = 252
dt = T / steps
lp = np.zeros((30, steps + 1))
for i in range(steps):
    m = rng.poisson(LAM * dt, 30)
    lp[:, i + 1] = lp[:, i] + (R - LAM * KBAR - 0.5 * VOL**2) * dt + VOL * math.sqrt(dt) * rng.standard_normal(30) \
                   + MUJ * m + DJ * np.sqrt(m) * rng.standard_normal(30)
merton = {"kbar": KBAR, "kbar_pct": KBAR * 100, "forward": float(ST.mean()), "forward_theory": S0 * math.exp(R * T),
          "share_no_jump": float((nj == 0).mean()), "t": np.linspace(0, 1, steps + 1)[::2].round(4).tolist(),
          "paths": (S0 * np.exp(lp[:, ::2])).round(2).tolist(),
          "ticks": [{"x": x, "label": lab} for x, lab in ((0, "0"), (0.25, "3m"), (0.5, "6m"), (0.75, "9m"), (1, "1y"))]}
assert abs(merton["forward"] / merton["forward_theory"] - 1) < 0.002
out["merton"] = merton
out["years"] = {"merton": 1976}
{k: v for k, v in merton.items() if not isinstance(v, list)}

## 5. One price, three ways

**The series.** Condition on the number of jumps $n$: given $N_T = n$, $\ln S_T$ is normal, so the call is a Black-Scholes price with volatility $\sigma_n = \sqrt{\sigma^2 + n\delta^2/T}$ and rate $r_n = r - \lambda\bar k + n\ln(1 + \bar k)/T$. Weight by the Poisson probabilities (with intensity $\lambda(1 + \bar k)$, after a change of measure that absorbs the $e^{n\ln(1+\bar k)}$ factor): `merton_call`.

**QuantLib.** The Python build of QuantLib 1.43 has no Merton engine, but Bates's model (Heston's stochastic variance plus Merton's jumps) with the variance frozen at $\sigma^2$ (a vol-of-vol of 0.001) is Merton's model.

**Monte Carlo** on the 400,000 simulated $S_T$.

In [ ]:
today = ql.Date(25, 9, 2026); ql.Settings.instance().evaluationDate = today
dc = ql.Actual365Fixed()
flat = lambda x: ql.YieldTermStructureHandle(ql.FlatForward(today, x, dc, ql.Continuous))
bates = ql.BatesProcess(flat(R), flat(0.0), ql.QuoteHandle(ql.SimpleQuote(S0)), VOL**2, 1.0, VOL**2, 1e-3, 0.0, LAM, MUJ, DJ)
opt = ql.VanillaOption(ql.PlainVanillaPayoff(ql.Option.Call, K), ql.EuropeanExercise(today + 365))
opt.setPricingEngine(ql.BatesEngine(ql.BatesModel(bates)))
pay = math.exp(-R * T) * np.maximum(ST - K, 0)
total_vol = math.sqrt(VOL**2 + LAM * (MUJ**2 + DJ**2))
price = {"series": float(merton_call(S0, K, T, R, VOL, LAM, MUJ, DJ)), "ql": opt.NPV(),
         "mc": float(pay.mean()), "mc_se": float(pay.std() / math.sqrt(n)),
         "total_vol_pct": total_vol * 100, "bs_total": float(bs_call(S0, K, T, R, total_vol)[0])}
assert abs(price["series"] - price["ql"]) < 1e-4 and abs(price["mc"] - price["series"]) < 3 * price["mc_se"]
out["price"] = price
pd.Series(price)

## 6. The smile

Price calls across strikes with Merton's formula, then ask which Black-Scholes volatility reproduces each price. A single volatility can't: the downward jumps fatten the left tail, so low strikes need a higher volatility. That is a **skew**, like the equity smile of Series 3, Episode 25.

In [ ]:
strikes = [70, 80, 90, 100, 105, 110, 120, 130]
smile = []
for k in strikes:
    p = float(merton_call(S0, k, T, R, VOL, LAM, MUJ, DJ))
    smile.append({"K": k, "price": p, "iv_pct": bs_implied_vol(p, S0, k, T, R) * 100})
assert smile[0]["iv_pct"] > smile[4]["iv_pct"] > smile[-1]["iv_pct"]
out["smile"] = {"rows": smile, "K": strikes, "iv": [round(s["iv_pct"], 4) for s in smile],
                "flat": [round(total_vol * 100, 4)] * len(strikes), "lo": smile[0]["iv_pct"], "atm": smile[4]["iv_pct"],
                "hi": smile[-1]["iv_pct"], "ticks": [{"x": k, "label": str(k)} for k in (70, 80, 90, 100, 110, 120, 130)]}
pd.DataFrame(smile)

## 7. Hedging breaks

Sell the call, delta-hedge with the share, rebalance $n$ times. In a world without jumps (geometric Brownian motion with the same total volatility, 19.7%), the error shrinks like $1/\sqrt n$ (Episode 22). With jumps it doesn't: a jump moves the share by a finite amount between two rebalancing dates, and the option's curvature turns that into a loss no delta can anticipate.

In [ ]:
def hedge(n, jumps, paths=2_000, muj=MUJ, dj=DJ, seed_=7):
    r_ = np.random.default_rng(seed_)
    kbar = math.exp(muj + 0.5 * dj**2) - 1
    tv = math.sqrt(VOL**2 + LAM * (muj**2 + dj**2))
    dt = T / n
    s = np.full(paths, S0)
    if jumps:
        price0 = float(merton_call(S0, K, T, R, VOL, LAM, muj, dj))
        delta = lambda x, tl: (merton_call(x + 0.01, K, tl, R, VOL, LAM, muj, dj, 20) - merton_call(x - 0.01, K, tl, R, VOL, LAM, muj, dj, 20)) / 0.02
    else:
        price0 = float(bs_call(S0, K, T, R, tv)[0])
        delta = lambda x, tl: bs_call(x, K, tl, R, tv)[1]
    d = delta(s, T)
    cash = price0 - d * s
    for i in range(n):
        if jumps:
            m = r_.poisson(LAM * dt, paths)
            s = s * np.exp((R - LAM * kbar - 0.5 * VOL**2) * dt + VOL * math.sqrt(dt) * r_.standard_normal(paths)
                           + muj * m + dj * np.sqrt(m) * r_.standard_normal(paths))
        else:
            s = s * np.exp((R - 0.5 * tv**2) * dt + tv * math.sqrt(dt) * r_.standard_normal(paths))
        cash *= math.exp(R * dt)
        left = T - (i + 1) * dt
        if left > 1e-12:
            nd = delta(s, left)
            cash -= (nd - d) * s
            d = nd
    return d * s + cash - np.maximum(s - K, 0)

rows = []
for n_ in (12, 52, 252):
    e0, e1 = hedge(n_, False), hedge(n_, True)
    rows.append({"rebalances": n_, "label": {12: "monthly", 52: "weekly", 252: "daily"}[n_],
                 "no_jumps": float(e0.std()), "jumps": float(e1.std())})
assert rows[0]["no_jumps"] / rows[2]["no_jumps"] > 3 and rows[2]["jumps"] > 3 * rows[2]["no_jumps"]
out["hedge"] = {"rows": rows, "paths": 2_000, "daily_no": rows[2]["no_jumps"], "daily_jump": rows[2]["jumps"],
                "monthly_no": rows[0]["no_jumps"], "monthly_jump": rows[0]["jumps"]}
pd.DataFrame(rows)

## 8. Incomplete markets

Episode 22: one Brownian motion and one traded share make the market complete, so every payoff has one price. Here there are two sources of risk (the Brownian motion and the jumps) and still one share. The martingale representation needs a second integrand, against the compensated jump process, with nothing traded to carry it. The market is **incomplete**: many risk-neutral measures fit the share's price, they differ in how they price jump risk, and the option's price depends on that choice. Merton's formula picks one (jump risk not priced). In practice, traders hedge jump risk with other options.

## 9. This episode's question

$P(N_1 = 0) = e^{-0.5} = 60.7\%$: in most years, this share doesn't jump at all.

In [ ]:
out["question"] = {"p_none_pct": math.exp(-LAM) * 100, "share_no_jump_sim_pct": merton["share_no_jump"] * 100}
out["question"]

## 10. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** Show that $E[N_t] = \mathrm{Var}(N_t) = \lambda t$ from the Poisson probabilities $P(N_t = k) = e^{-\lambda t}(\lambda t)^k/k!$.

<details><summary>Answer</summary>

$E[N] = \sum k\,e^{-m}m^k/k! = m\sum e^{-m}m^{k-1}/(k-1)! = m$ with $m = \lambda t$. Similarly $E[N(N - 1)] = m^2$, so $\mathrm{Var}(N) = m^2 + m - m^2 = m$.

</details>

**Exercise 2 (By hand).** Why does Merton's drift include $-\lambda\bar k$?

<details><summary>Answer</summary>

Over $dt$ a jump arrives with probability $\lambda\,dt$ and multiplies $S$ by $J$, adding $E[J - 1] = \bar k$ per jump: an expected return of $\lambda\bar k\,dt$. For $e^{-rt}S$ to be a martingale the total expected return must be $r\,dt$, so the drift is $r - \lambda\bar k$.

</details>

**Exercise 3 (By hand).** What does Merton's formula reduce to when $\lambda = 0$? And when $\delta = 0$ and $\mu_J = 0$?

<details><summary>Answer</summary>

$\lambda = 0$: only the $n = 0$ term survives, which is Black-Scholes with volatility $\sigma$. $\delta = \mu_J = 0$: $J = 1$, the jumps don't move the price, $\bar k = 0$, and every term is the same Black-Scholes price; the weights sum to 1.

</details>

**Exercise 4 (In code).** Recompute the smile with a positive mean jump, $\mu_J = +10\%$. Which way does it tilt?

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
iv = {k: bs_implied_vol(float(merton_call(S0, k, T, R, VOL, LAM, 0.10, DJ)), S0, k, T, R) for k in (80, 100, 120)}
assert iv[120] > iv[80]
print({k: f"{v:.2%}" for k, v in iv.items()}, "- upward jumps fatten the right tail: high strikes now need more volatility")

**Exercise 5 (In code).** Hedge daily with smaller jumps (mean log jump −5%, jump volatility 5%). How does the error floor change?

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
small = float(hedge(252, True, muj=-0.05, dj=0.05).std())
assert small < out["hedge"]["daily_jump"]
print(f"daily hedging error: {out['hedge']['daily_jump']:.2f} with the base jumps, {small:.2f} with small jumps")

## 11. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")